## 03\_rouen\_zones\_5min

# 03 · Zones accessibles à 5 minutes à pied

Calcul des couples (origine, destination) distants de **400 m au plus sur le réseau piéton** OpenStreetMap, soit 5 minutes à 4,8 km/h :

- options de transport accessibles depuis chaque carreau habité (objectifs 1 et 2) ;
- carreaux desservis par un équipement implanté au centre de chaque carreau candidat (objectif 3).

Le réseau exclut les voies interdites ou hostiles aux piétons (autoroutes, voies rapides, sites propres bus). Les distances sont calculées par l'algorithme de Dijkstra, à partir du nœud du réseau le plus proche de chaque point.

**Variante** (`MODE_VOL_OISEAU = True`) : disque de 330 m à vol d'oiseau, équivalent à 400 m de marche avec un coefficient de détour de 1,2.

**Entrées :** `carreaux_rouen`, `options_rouen` (notebook 02) ; réseau OpenStreetMap Geofabrik dans `Files/brut/osm`.

**Sorties :** tables `zones_5min` (option, carreau) et `couverture_candidats` (emplacement candidat, carreau desservi).

In [ ]:
# 1. Dépendances
%pip install -q geopandas pyogrio py7zr networkx

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# 2. Outils communs
import os, re, glob, shutil, tempfile, zipfile, warnings
import numpy as np
import pandas as pd
from pyproj import Transformer
warnings.filterwarnings("ignore")

BASE = "/lakehouse/default/Files"
if not os.path.isdir(BASE):
    raise RuntimeError("Lakehouse par défaut non attaché : attacher le Lakehouse mobilite au notebook.")
BRUT = os.path.join(BASE, "brut")
TRAITE = os.path.join(BASE, "traite")
CARTES = os.path.join(TRAITE, "cartes")
os.makedirs(CARTES, exist_ok=True)

# Conversions entre systèmes de coordonnées : 3035 = grille INSEE, 2154 = Lambert-93 (mètres), 4326 = GPS
T_3035_2154 = Transformer.from_crs(3035, 2154, always_xy=True)
T_3035_4326 = Transformer.from_crs(3035, 4326, always_xy=True)
T_4326_3035 = Transformer.from_crs(4326, 3035, always_xy=True)
T_4326_2154 = Transformer.from_crs(4326, 2154, always_xy=True)

def _copier_csv(df, chemin):
    """Écrit le CSV en local, puis le copie dans le Lakehouse. Écrire directement dans OneLake échoue parfois
    quand le fichier existe déjà (« Unable to rename file »)."""
    fd, tmp = tempfile.mkstemp(suffix=".csv")
    os.close(fd)
    try:
        df.to_csv(tmp, index=False)
        if os.path.exists(chemin):
            os.remove(chemin)
        shutil.copyfile(tmp, chemin)
    finally:
        os.remove(tmp)

def _adresse_table(nom):
    """Adresse OneLake de la table dans le Lakehouse attaché, dans le schéma dbo s'il existe."""
    import notebookutils
    ctx = notebookutils.runtime.context
    espace = ctx.get("defaultLakehouseWorkspaceId") or ctx.get("currentWorkspaceId")
    lakehouse = ctx.get("defaultLakehouseId")
    if not (espace and lakehouse):
        raise RuntimeError("Lakehouse par défaut introuvable : attacher le Lakehouse mobilite au notebook")
    schema = "/dbo" if os.path.isdir("/lakehouse/default/Tables/dbo") else ""
    return f"abfss://{espace}@onelake.dfs.fabric.microsoft.com/{lakehouse}/Tables{schema}/{nom}"

def _ecrire_table(df, nom):
    """Table Delta lue par Power BI. Noyau Spark : saveAsTable. Noyau Python : bibliothèque deltalake."""
    try:
        spark  # n'existe que dans le noyau Spark
    except NameError:
        import pyarrow as pa
        import notebookutils
        from deltalake import write_deltalake
        donnees = pa.Table.from_pandas(df, preserve_index=False)
        options = {"bearer_token": notebookutils.credentials.getToken("storage"), "use_fabric_endpoint": "true"}
        try:
            write_deltalake(_adresse_table(nom), donnees, mode="overwrite", schema_mode="overwrite",
                            storage_options=options)
        except TypeError:   # anciennes versions de deltalake
            write_deltalake(_adresse_table(nom), donnees, mode="overwrite", overwrite_schema=True,
                            storage_options=options)
        return
    spark.createDataFrame(df).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(nom)

def enregistrer(df, nom):
    """Enregistre un tableau en CSV dans Files/traite (relu par les autres notebooks)
    et en table du Lakehouse (lue par Power BI), avec le noyau Python comme avec le noyau Spark."""
    df = df.reset_index(drop=True).copy()
    _copier_csv(df, os.path.join(TRAITE, nom + ".csv"))
    for c in df.columns:
        if not (pd.api.types.is_numeric_dtype(df[c]) or pd.api.types.is_bool_dtype(df[c])):
            df[c] = df[c].map(lambda v: None if v is None or (isinstance(v, float) and np.isnan(v)) else str(v))
    try:
        _ecrire_table(df, nom)
        print(f"   table {nom} : {len(df)} lignes (copie dans Files/traite/{nom}.csv)")
    except Exception as e:
        print(f"   {nom}.csv : {len(df)} lignes. ATTENTION, table non créée pour Power BI : {e}")

def nb(n):
    """Écrit un nombre avec des espaces entre les milliers : 12 345."""
    return f"{n:,.0f}".replace(",", " ")

def lire(nom):
    """Relit un tableau enregistré par un autre notebook."""
    chemin = os.path.join(TRAITE, nom + ".csv")
    if not os.path.exists(chemin):
        raise FileNotFoundError(f"{nom}.csv introuvable dans Files/traite : exécuter d'abord le notebook qui le produit.")
    return pd.read_csv(chemin, dtype={"idcar": str, "commune": str, "idcar_1km": str, "candidat": str})

COLONNES_FILOSOFI = ["idcar_200m", "idcar_1km", "i_est_200", "lcog_geo", "ind", "men", "men_pauv", "men_1ind",
                     "men_5ind", "men_prop", "men_fmp", "ind_snv", "men_surf", "men_coll", "men_mais", "log_av45",
                     "log_45_70", "log_70_90", "log_ap90", "log_inc", "log_soc", "ind_0_3", "ind_4_5", "ind_6_10",
                     "ind_11_17", "ind_18_24", "ind_25_39", "ind_40_54", "ind_55_64", "ind_65_79", "ind_80p", "ind_inc"]
TEXTE_FILOSOFI = ["idcar_200m", "idcar_1km", "lcog_geo"]

def charger_filosofi():
    """Carreaux de 200 m de métropole (CSV ou GeoPackage), avec leur centre en coordonnées 3035."""
    csvs = sorted(glob.glob(os.path.join(BRUT, "filosofi", "**", "*.csv"), recursive=True),
                  key=os.path.getsize, reverse=True)
    csvs = [c for c in csvs if "met" in os.path.basename(c).lower()] or csvs
    if csvs:
        source = csvs[0]
        with open(source, encoding="utf-8", errors="replace") as f:
            entete = f.readline()
        sep = ";" if entete.count(";") > entete.count(",") else ","
        noms = [c.strip().strip('"') for c in entete.strip().split(sep)]
        utiles = [c for c in noms if c.lower() in COLONNES_FILOSOFI]
        df = pd.read_csv(source, sep=sep, usecols=utiles,
                         dtype={c: str for c in utiles if c.lower() in TEXTE_FILOSOFI})
    else:
        gpkgs = [g for g in glob.glob(os.path.join(BRUT, "**", "*.gpkg"), recursive=True)
                 if "carreaux" in os.path.basename(g).lower() and "met" in os.path.basename(g).lower()]
        if not gpkgs:
            raise FileNotFoundError("Carreaux Filosofi introuvables dans Files/brut/filosofi (notebook 01).")
        import pyogrio
        source = gpkgs[0]
        df = pyogrio.read_dataframe(source, read_geometry=False)
    df.columns = [c.lower() for c in df.columns]
    df = df[[c for c in COLONNES_FILOSOFI if c in df.columns]].copy()
    for c in df.columns:
        if c not in TEXTE_FILOSOFI and not pd.api.types.is_numeric_dtype(df[c]):   # décimales avec une virgule
            df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", ".", regex=False), errors="coerce")
    nombres = [c for c in df.columns if c not in TEXTE_FILOSOFI]
    df[nombres] = df[nombres].fillna(0)
    # L'identifiant INSPIRE donne le coin sud-ouest du carreau en mètres : CRS3035RES200mN<nord>E<est>
    ne = df["idcar_200m"].astype(str).str.extract(r"CRS3035RES200mN(\d+)E(\d+)").astype(float)
    df["x3035"], df["y3035"] = ne[1] + 100, ne[0] + 100
    df = df.dropna(subset=["x3035", "y3035"])
    df["ix"] = ((df["x3035"] - 100) // 200).astype("int64")
    df["iy"] = ((df["y3035"] - 100) // 200).astype("int64")
    df["commune"] = df["lcog_geo"].astype(str).str[:5]
    df = df.rename(columns={"idcar_200m": "idcar"}).reset_index(drop=True)
    print(f"Filosofi : {nb(len(df))} carreaux de métropole lus ({os.path.basename(source)})")
    return df

def trouver_admin_express():
    """GeoPackage ADMIN EXPRESS : déjà extrait, ou extrait de l'archive .7z déposée dans Files/brut."""
    gpkgs = [g for g in glob.glob(os.path.join(BRUT, "**", "*.gpkg"), recursive=True)
             if "ADE" in os.path.basename(g).upper() or "ADMIN" in os.path.basename(g).upper()]
    if gpkgs:
        return gpkgs[0]
    archives = [a for a in glob.glob(os.path.join(BRUT, "**", "*.7z"), recursive=True) if "ADMIN" in a.upper()]
    if not archives:
        raise FileNotFoundError("Contours introuvables : fichiers IGN du notebook 01 ou archive ADMIN-EXPRESS (.7z) attendus dans Files/brut.")
    import py7zr
    with py7zr.SevenZipFile(sorted(archives)[-1], "r") as z:
        z.extractall(os.path.join(BRUT, "ign"))
    return trouver_admin_express()

def charger_territoire(court, mot):
    """Contour de l'intercommunalité (Lambert-93) et ses communes."""
    import geopandas as gpd
    f_epci = os.path.join(BRUT, f"epci_{court}.geojson")
    f_communes = os.path.join(BRUT, f"communes_{court}.geojson")
    if os.path.exists(f_epci) and os.path.exists(f_communes):
        epci = gpd.read_file(f_epci).to_crs(2154)
        communes = gpd.read_file(f_communes).to_crs(2154)
    else:
        gpkg = trouver_admin_express()
        epci = gpd.read_file(gpkg, layer="epci")
        epci = epci[epci["nom_officiel"].str.contains(mot, case=False, na=False)].to_crs(2154)
        siren = str(epci.iloc[0]["code_siren"])
        communes = gpd.read_file(gpkg, layer="commune")
        communes = communes[communes["codes_siren_des_epci"].astype(str).str.contains(siren, na=False)].to_crs(2154)
    contour = epci.geometry.iloc[0]
    communes = communes[["code_insee", "nom_officiel", "geometry"]].rename(
        columns={"code_insee": "commune", "nom_officiel": "commune_nom"})
    print(f"{epci.iloc[0]['nom_officiel']} : {len(communes)} communes, {contour.area / 1e6:.0f} km²")
    return contour, communes

def dans_contour(contour, x, y):
    """Vrai pour les points (Lambert-93) situés dans le contour."""
    import shapely
    xmin, ymin, xmax, ymax = contour.bounds
    resultat = np.zeros(len(x), dtype=bool)
    cadre = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)
    shapely.prepare(contour)
    resultat[cadre] = shapely.contains_xy(contour, x[cadre], y[cadre])
    return resultat

In [ ]:
# 3. Réglages
MODE_VOL_OISEAU = False        # True : cercle de 330 m à vol d'oiseau au lieu du réseau piéton
MARCHE_M = 400        # 5 minutes à pied
VOL_OISEAU_M = 330    # équivalent à vol d'oiseau (400 m de marche avec 20 % de détour)

In [ ]:
# 4. Carreaux et options de Rouen (notebook 02)
carreaux = lire("carreaux_rouen")
options = lire("options_rouen")
contour, _ = charger_territoire("rouen", "Rouen")
print(f"{len(carreaux)} carreaux habités, {len(options)} options de transport")

Métropole Rouen Normandie : 71 communes, 664 km²
5727 carreaux habités, 1288 options de transport


In [ ]:
# 5. Réseau piéton OpenStreetMap (fichier Geofabrik déposé dans Files/brut/osm)
import shapely, pyogrio
import networkx as nx
from scipy.spatial import cKDTree

def trouver_osm():
    dossier = os.path.join(BRUT, "osm")
    fichiers = [f for f in glob.glob(os.path.join(dossier, "**", "*.gpkg"), recursive=True) if os.path.isfile(f)]
    if fichiers:
        return fichiers[0]
    for archive in glob.glob(os.path.join(dossier, "**", "*.zip"), recursive=True):
        with zipfile.ZipFile(archive) as z:
            membres = [m for m in z.namelist() if m.lower().endswith(".gpkg") and not m.endswith("/")]
            if membres:
                print(f"Décompression de {os.path.basename(membres[0])} (une minute environ)...")
                z.extract(membres[0], dossier)
                return os.path.join(dossier, membres[0])
    return None

EXCLUS = {"motorway", "motorway_link", "trunk", "trunk_link", "busway"}   # voies interdites ou hostiles aux piétons
if not MODE_VOL_OISEAU:
    chemin_osm = trouver_osm()
    if chemin_osm is None:
        print("Fichier OpenStreetMap introuvable dans Files/brut/osm : calcul à vol d'oiseau.")
        MODE_VOL_OISEAU = True
if not MODE_VOL_OISEAU:
    import geopandas as gpd
    cadre = gpd.GeoSeries([contour.buffer(1000)], crs=2154).to_crs(4326).total_bounds
    routes = pyogrio.read_dataframe(chemin_osm, layer="gis_osm_roads_free", bbox=tuple(cadre), columns=["fclass"])
    routes = routes[~routes["fclass"].isin(EXCLUS)].to_crs(2154)
    coords, ligne = shapely.get_coordinates(routes.geometry.values, return_index=True)
    cle = np.round(coords * 10).astype(np.int64)                     # deux points identiques au décimètre = même nœud
    noeud, uniques = pd.factorize(cle[:, 0] * 100_000_000 + cle[:, 1])
    suite = ligne[1:] == ligne[:-1]
    a, b = noeud[:-1][suite], noeud[1:][suite]
    longueur = np.hypot(*(coords[1:][suite] - coords[:-1][suite]).T)
    G = nx.Graph()
    G.add_weighted_edges_from(zip(a.tolist(), b.tolist(), longueur.tolist()), weight="m")
    xy = np.zeros((len(uniques), 2))
    xy[noeud] = coords
    plus_grand = max(len(c) for c in nx.connected_components(G))
    print(f"Réseau piéton : {len(routes)} voies, {G.number_of_nodes()} nœuds, "
          f"{100 * plus_grand / G.number_of_nodes():.1f} % reliés entre eux")

Réseau piéton : 75689 voies, 401402 nœuds, 99.6 % reliés entre eux


In [ ]:
# 6. Ce qui est à 5 minutes de chaque carreau
from collections import defaultdict
paires_carreaux, paires_options = [], []
cx, cy = carreaux["x"].values, carreaux["y"].values
ox, oy = options["x"].values, options["y"].values

if MODE_VOL_OISEAU:
    arbre_c, arbre_o = cKDTree(np.column_stack([cx, cy])), cKDTree(np.column_stack([ox, oy]))
    for i, voisins in enumerate(arbre_c.query_ball_point(np.column_stack([cx, cy]), VOL_OISEAU_M)):
        paires_carreaux += [(i, j) for j in voisins]
    for i, voisins in enumerate(arbre_c.query_ball_point(np.column_stack([ox, oy]), VOL_OISEAU_M)):
        paires_options += [(i, j) for j in voisins]
    methode = f"cercle de {VOL_OISEAU_M} m à vol d'oiseau"
else:
    arbre_n = cKDTree(xy)
    d_c, n_c = arbre_n.query(np.column_stack([cx, cy]))     # nœud le plus proche de chaque centre de carreau
    d_o, n_o = arbre_n.query(np.column_stack([ox, oy]))     # et de chaque option
    carreaux_du_noeud, options_du_noeud = defaultdict(list), defaultdict(list)
    for j, (n, d) in enumerate(zip(n_c, d_c)):
        carreaux_du_noeud[n].append((j, d))
    for k, (n, d) in enumerate(zip(n_o, d_o)):
        options_du_noeud[n].append((k, d))
    for i in range(len(carreaux)):
        reste = MARCHE_M - d_c[i]
        if reste <= 0:
            continue
        atteints = nx.single_source_dijkstra_path_length(G, int(n_c[i]), cutoff=reste, weight="m")
        for n, d in atteints.items():
            for j, dj in carreaux_du_noeud.get(n, ()):
                if d + dj <= reste:
                    paires_carreaux.append((i, j))
            for k, dk in options_du_noeud.get(n, ()):
                if d + dk <= reste:
                    paires_options.append((k, i))
        if i % 2000 == 0:
            print(f"   {i} carreaux traités sur {len(carreaux)}")
    methode = f"{MARCHE_M} m sur le réseau piéton OpenStreetMap"

# Un carreau est toujours desservi par une station placée en son centre
paires_carreaux += [(i, i) for i in range(len(carreaux))]
couverture = pd.DataFrame(paires_carreaux, columns=["i", "j"]).drop_duplicates()
couverture = pd.DataFrame({"candidat": carreaux["idcar"].values[couverture["i"]], "idcar": carreaux["idcar"].values[couverture["j"]]})
zones = pd.DataFrame(paires_options, columns=["k", "i"]).drop_duplicates()
zones = pd.DataFrame({"option_id": options["option_id"].values[zones["k"]], "idcar": carreaux["idcar"].values[zones["i"]]})
print(f"Méthode : {methode}")
print(f"Une station placée au centre d'un carreau dessert en moyenne {couverture.groupby('candidat').size().mean():.1f} carreaux")
print(f"Une option dessert en moyenne {zones.groupby('option_id').size().mean():.1f} carreaux ; "
      f"{(~options['option_id'].isin(zones['option_id'])).sum()} options ne desservent aucun carreau habité")

   0 carreaux traités sur 5727
   2000 carreaux traités sur 5727
   4000 carreaux traités sur 5727
Méthode : 400 m sur le réseau piéton OpenStreetMap
Une station placée au centre d'un carreau dessert en moyenne 3.2 carreaux
Une option dessert en moyenne 4.6 carreaux ; 45 options ne desservent aucun carreau habité


In [ ]:
# 7. Enregistrer
print("Enregistrement :")
enregistrer(zones, "zones_5min")
enregistrer(couverture, "couverture_candidats")
desservis = carreaux["idcar"].isin(zones["idcar"])
print(f"\nAperçu : {100 * carreaux.loc[desservis, 'population'].sum() / carreaux['population'].sum():.0f} % des habitants "
      f"ont au moins une option à 5 minutes à pied ({methode}).")

Enregistrement :


[2026-10-21T08:20:48Z WARN  delta_kernel::listed_log_files] Found file 00000000000000000000.parquet with unknown file type Unknown at version 0
[2026-10-21T08:20:49Z WARN  delta_kernel::listed_log_files] Found file 00000000000000000000.parquet with unknown file type Unknown at version 0
